# Bronze — trust metadata

`landing.trusts_raw` → `bronze.trusts`. Every column cast to STRING, nothing else.

Landing already holds this as text, so the casts are a no-op here **by design** —
the point is that all three Bronze tables honour one contract, with no exceptions
to remember.

Expected: **120 rows**, same as Landing.

In [0]:
%sql
-- Explicit column list rather than SELECT * : it documents the source contract, so
-- a column appearing or vanishing upstream shows up as an error, not a surprise.
CREATE OR REPLACE TABLE `index-vs-trust-pipeline`.bronze.trusts AS
SELECT
  CAST(trust_name       AS STRING) AS trust_name,
  CAST(ticker           AS STRING) AS ticker,
  CAST(aic_sector       AS STRING) AS aic_sector,
  CAST(source_url       AS STRING) AS source_url,
  CAST(manager          AS STRING) AS manager,
  CAST(management_group AS STRING) AS management_group,
  CAST(notes            AS STRING) AS notes
FROM `index-vs-trust-pipeline`.landing.trusts_raw;

## Verification

In [0]:
%sql
-- Bronze must reject nothing, so the two counts have to be equal.
SELECT
  (SELECT COUNT(*) FROM `index-vs-trust-pipeline`.landing.trusts_raw) AS landing_rows,
  (SELECT COUNT(*) FROM `index-vs-trust-pipeline`.bronze.trusts)      AS bronze_rows,
  (SELECT COUNT(*) FROM `index-vs-trust-pipeline`.bronze.trusts
     WHERE ticker = '')                                               AS blank_tickers;

Expect **120 / 120 / 2**. The two blank tickers (Island Innovation, Witan) surviving
is the proof that Bronze dropped nothing.

In [0]:
%sql
-- Every column must be STRING. Any other type here is a bug.
DESCRIBE TABLE `index-vs-trust-pipeline`.bronze.trusts;